# Imports

In [53]:
import re
import optuna
import pickle

import numpy as np
import pandas as pd

from sklearn import set_config

from scipy.stats import entropy, rankdata

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import StackingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.model_selection import StratifiedKFold

from feature_engine.selection import DropFeatures

## Utils

In [2]:
set_config(enable_metadata_routing=True)

In [13]:
class StackingFeatureEngineer(BaseEstimator, TransformerMixin):

    def __init__(
        self,
        add_rank_features=True,
        add_statistical_features=True,
        add_interaction_features=True,
        add_difference_features=True,
        add_entropy_features=True,
        add_threshold_features=True,
        confidence_threshold=0.9,
        uncertainty_low=0.4,
        uncertainty_high=0.6,
    ):

        self.add_rank_features = add_rank_features
        self.add_statistical_features = add_statistical_features
        self.add_interaction_features = add_interaction_features
        self.add_difference_features = add_difference_features
        self.add_entropy_features = add_entropy_features
        self.add_threshold_features = add_threshold_features

        self.confidence_threshold = confidence_threshold
        self.uncertainty_low = uncertainty_low
        self.uncertainty_high = uncertainty_high

    def fit(self, X, y=None):

        X = self._to_dataframe(X)

        self.feature_names_in_ = X.columns.tolist()

        return self

    def transform(self, X):

        X = self._to_dataframe(
            X,
            columns=getattr(self, "feature_names_in_", None)
        )

        features = []

        # =====================================================
        # FEATURES ORIGINAIS
        # =====================================================

        base_probs = X.copy()

        features.append(base_probs)

        # =====================================================
        # FEATURES ESTATÍSTICAS
        # =====================================================

        if self.add_statistical_features:

            stat_df = pd.DataFrame(index=X.index)

            stat_df["prob_mean"] = X.mean(axis=1)
            stat_df["prob_std"] = X.std(axis=1)
            stat_df["prob_min"] = X.min(axis=1)
            stat_df["prob_max"] = X.max(axis=1)
            stat_df["prob_range"] = (
                stat_df["prob_max"] -
                stat_df["prob_min"]
            )

            stat_df["prob_median"] = X.median(axis=1)

            features.append(stat_df)

        # =====================================================
        # ENTROPIA
        # =====================================================

        if self.add_entropy_features:

            ent_df = pd.DataFrame(index=X.index)

            probs_norm = (
                X.div(X.sum(axis=1) + 1e-12, axis=0)
            )

            ent_df["prediction_entropy"] = probs_norm.apply(
                lambda row: entropy(row + 1e-12),
                axis=1
            )

            features.append(ent_df)

        # =====================================================
        # RANK FEATURES
        # =====================================================

        if self.add_rank_features:

            rank_df = pd.DataFrame(index=X.index)

            ranks = np.apply_along_axis(
                rankdata,
                axis=1,
                arr=X.values
            )

            for i, col in enumerate(X.columns):

                rank_df[f"{col}_rank"] = ranks[:, i]

            features.append(rank_df)

        # =====================================================
        # AGREEMENT FEATURES
        # =====================================================

        vote_df = pd.DataFrame(index=X.index)

        preds = (X > 0.5).astype(int)

        vote_df["vote_sum"] = preds.sum(axis=1)
        vote_df["vote_mean"] = preds.mean(axis=1)

        vote_df["all_agree"] = (
            preds.nunique(axis=1) == 1
        ).astype(int)

        vote_df["disagreement"] = (
            preds.nunique(axis=1) > 1
        ).astype(int)

        features.append(vote_df)

        # =====================================================
        # THRESHOLD FEATURES
        # =====================================================

        if self.add_threshold_features:

            thresh_df = pd.DataFrame(index=X.index)

            thresh_df["high_conf_models"] = (
                X > self.confidence_threshold
            ).sum(axis=1)

            thresh_df["uncertain_models"] = (
                (
                    X > self.uncertainty_low
                ) &
                (
                    X < self.uncertainty_high
                )
            ).sum(axis=1)

            features.append(thresh_df)

        # =====================================================
        # INTERAÇÕES
        # =====================================================

        if self.add_interaction_features:

            inter_df = pd.DataFrame(index=X.index)

            cols = X.columns.tolist()

            for i in range(len(cols)):
                for j in range(i + 1, len(cols)):

                    c1 = cols[i]
                    c2 = cols[j]

                    inter_df[f"{c1}_x_{c2}"] = (
                        X[c1] * X[c2]
                    )

            features.append(inter_df)

        # =====================================================
        # DIFERENÇAS
        # =====================================================

        if self.add_difference_features:

            diff_df = pd.DataFrame(index=X.index)

            cols = X.columns.tolist()

            for i in range(len(cols)):
                for j in range(i + 1, len(cols)):

                    c1 = cols[i]
                    c2 = cols[j]

                    diff_df[f"{c1}_minus_{c2}"] = (
                        X[c1] - X[c2]
                    )

            features.append(diff_df)

        # =====================================================
        # CONCAT FINAL
        # =====================================================

        X_meta = pd.concat(features, axis=1)

        return X_meta

    @staticmethod
    def _to_dataframe(X, columns=None):

        if isinstance(X, pd.DataFrame):
            return X.copy()

        X = np.asarray(X)

        if columns is None:
            columns = [
                f"feature_{i}"
                for i in range(X.shape[1])
            ]

        return pd.DataFrame(X, columns=columns)

# Loading Dataset

In [4]:
X_train = pd.read_parquet('../data/processed/X_train_stacking2.parquet')
y_train = pd.read_parquet('../data/processed/y_train.parquet')

X_test = pd.read_parquet('../data/processed/X_test_stacking2.parquet')

In [5]:
X_train.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,voting_lgbm_cat_xgb_hist_proba,stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba
0,0.802607,0.891870,0.895659,0.883947,0.808497,0.887454,0.868510,0.826590
1,0.649405,0.831360,0.829854,0.779541,0.689888,0.519966,0.772514,0.801335
2,0.505904,0.851809,0.751522,0.758911,0.558520,0.209855,0.717012,0.782757
3,0.001713,0.005910,0.007057,0.007298,0.002275,0.000000,0.005494,0.502476
4,0.392205,0.565506,0.489033,0.482967,0.414144,0.297043,0.482416,0.703165


In [6]:
X_test.head()

,lgbm_proba,cat_proba,xgb_proba,hist_proba,lg_proba,knn_proba,voting_lgbm_cat_xgb_hist_proba,stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba
0,0.004591,0.031986,0.022774,0.020057,0.006419,0.000000,0.019849,0.509303
1,0.003032,0.011817,0.014957,0.018887,0.063872,0.117213,0.012173,0.505573
2,0.003777,0.011641,0.014307,0.011497,0.004517,0.000000,0.010304,0.504827
3,0.163131,0.544040,0.585898,0.485388,0.655870,0.099916,0.444558,0.696136
4,0.872172,0.967646,0.959451,0.959356,0.840450,1.000000,0.939648,0.852833


# Feature Engineering

## Feature Creation

In [14]:
sfe = StackingFeatureEngineer(
    add_rank_features=True,
    add_statistical_features=True,
    add_entropy_features=True,
    add_threshold_features=True,
    add_interaction_features=False,
    add_difference_features=False,
)

X_train_enc = sfe.fit_transform(X_train)
X_test_enc = sfe.transform(X_test)

In [48]:
X_train_enc.to_parquet('../data/processed/X_train_stacking3.parquet')
X_test_enc.to_parquet('../data/processed/X_test_stacking3.parquet')

## Feature Scaling

In [51]:
column_transformer = ColumnTransformer([
    (
        'standard_scaler', 
        StandardScaler(), 
        [
            'lgbm_proba', 'cat_proba', 'xgb_proba', 'hist_proba', 'lg_proba', 'knn_proba', 'voting_lgbm_cat_xgb_hist_proba', 
            'stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba', 'prob_mean', 'prob_max', 'prob_median', 'prediction_entropy',
            'cat_proba_rank', 'xgb_proba_rank', 'hist_proba_rank', 'lg_proba_rank', 'knn_proba_rank', 'voting_lgbm_cat_xgb_hist_proba_rank',
            'vote_sum', 'vote_mean'
        ]
    ),
    (
        'robust_scaler', 
        RobustScaler(), 
        [
            'prob_std', 'prob_min', 'prob_range', 'lgbm_proba_rank', 'stacking_lg_knn_voting_lgbm_cat_xgb_hist_proba_rank', 
            'all_agree', 'disagreement', 'high_conf_models', 'uncertain_models'
        ]
    ),
], remainder="passthrough")

## Feature Selection

In [58]:
model = make_pipeline(
    column_transformer,
    LogisticRegression(random_state=42, verbose=0, class_weight='balanced', max_iter=500)
).fit(X_train_enc, y_train.PitNextLap)

perm_result = permutation_importance(
    estimator=model, 
    X=X_train_enc, 
    y=y_train.PitNextLap, 
    n_jobs=-1, 
    scoring='roc_auc'
)

importance_df = pd.DataFrame({
    "feature": X_train_enc.columns.tolist(),
    "importance_mean": perm_result.importances_mean,
    "importance_std": perm_result.importances_std
}).sort_values(by="importance_mean", ascending=False)

features_to_drop = importance_df.query("importance_mean <= 0").feature.tolist()

# Machine Learning

In [70]:
def objective(trial, X, y):

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    aucs = []

    params = {
        "solver": trial.suggest_categorical("solver", ["saga"]),
        "C": trial.suggest_float("C", 1e-5, 100, log=True),
        "l1_ratio": trial.suggest_float("l1_ratio", 0.0, 1.0),
        "class_weight": trial.suggest_categorical("class_weight", [None, "balanced"]),
        "fit_intercept": trial.suggest_categorical("fit_intercept", [True, False]),
        "tol": trial.suggest_float("tol", 1e-6, 1e-2, log=True),
        "max_iter": trial.suggest_int("max_iter", 1000, 5000)
    }

    for fold, (train_idx, valid_idx) in enumerate(cv.split(X, y)):

        X_train, X_valid = X.iloc[train_idx, :], X.iloc[valid_idx, :]
        y_train, y_valid = y.iloc[train_idx, 0], y.iloc[valid_idx, 0]

        model = make_pipeline(
            column_transformer,
            LogisticRegression(**params)
        ).fit(X_train, y_train)

        proba = model.predict_proba(X_valid)[:, 1]

        auc = roc_auc_score(y_valid, proba)
        aucs.append(auc)

        trial.report(np.mean(aucs), step=fold)

        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

    return np.mean(aucs)

study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(seed=42), pruner=optuna.pruners.MedianPruner(n_warmup_steps=2))
study.optimize(lambda trial: objective(trial, X_train_enc, y_train), n_trials=500, n_jobs=-1, show_progress_bar=True)


print("Best trial score:")
print(study.best_trial.value)

print("\nBest params:")
print(study.best_trial.params)

[I 2026-05-18 17:06:10,895] A new study created in memory with name: no-name-10b72739-9c03-4772-b0ab-295081f2ca89
Best trial: 11. Best value: 0.949527:   0%|          | 1/500 [00:26<3:41:22, 26.62s/it]

[I 2026-05-18 17:06:37,509] Trial 11 finished with value: 0.9495267260133892 and parameters: {'solver': 'saga', 'C': 0.01165368745794059, 'l1_ratio': 0.2504298365789678, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 0.007948327613774918, 'max_iter': 4364}. Best is trial 11 with value: 0.9495267260133892.


Best trial: 6. Best value: 0.949531:   0%|          | 2/500 [00:33<2:04:21, 14.98s/it] 

[I 2026-05-18 17:06:44,342] Trial 6 finished with value: 0.949530914269172 and parameters: {'solver': 'saga', 'C': 0.000879553907442171, 'l1_ratio': 0.21547588979229515, 'class_weight': 'balanced', 'fit_intercept': True, 'tol': 1.0453882946161282e-05, 'max_iter': 1518}. Best is trial 6 with value: 0.949530914269172.


Best trial: 8. Best value: 0.949561:   1%|          | 3/500 [00:37<1:23:29, 10.08s/it]

...

[I 2026-05-18 18:47:24,246] Trial 499 finished with value: 0.9495816138427398 and parameters: {'solver': 'saga', 'C': 0.0009141305312417677, 'l1_ratio': 0.30217255955316413, 'class_weight': None, 'fit_intercept': True, 'tol': 0.00015332526115608615, 'max_iter': 3682}. Best is trial 168 with value: 0.9495820845736207.


Best trial: 168. Best value: 0.949582:  99%|█████████▊| 493/500 [1:41:13<00:21,  3.11s/it]

[I 2026-05-18 18:47:24,510] Trial 498 finished with value: 0.9495770225145985 and parameters: {'solver': 'saga', 'C': 0.0013651608231249844, 'l1_ratio': 0.24240658726993086, 'class_weight': None, 'fit_intercept': True, 'tol': 0.00015058059707508286, 'max_iter': 3323}. Best is trial 168 with value: 0.9495820845736207.


/home/junior/Documentos/GitHub/prediction-f1-pit-stops-kaggle-competition/.venv/lib/python3.13/site-packages/sklearn/linear_model/_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(
Best trial: 168. Best value: 0.949582:  99%|█████████▊| 493/500 [1:42:35<01:27, 12.49s/it]


KeyboardInterrupt: 

In [ ]:
stacking_lg = LogisticRegression(**study.best_params).fit(X_train_enc, y_train.PitNextLap)

# Submission

In [ ]:
submission = pd.read_csv('../data/raw/sample_submission.csv')

In [ ]:
submission['PitNextLap'] = stacking_lg.predict_proba(X_test)[:, 1]

In [ ]:
submission.to_csv('../data/submission/submission.csv', index=False)